# Aula 03 — Neurônio Artificial, Perceptron e 1ª Rede Neural

**Disciplina:** Inteligência Artificial  
**Professor:** Marcelo Batista  

🎯 OBJETIVOS

- Analista de Banco: Entender pesos/viés com analogia real (emprego + renda)
- Neurônio Perceptron do Zero: Construir em Python puro

✅ Regra de Ouro: Dados → Treino → Teste (sem vazamento!)


In [10]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
np.random.seed(SEED)
print("✅ Setup OK!")

✅ Setup OK!


## 🧠 O neurônio artificial (matemática mínima)

Um neurônio artificial pode ser visto como:

$
y = f\left(\sum_i x_i w_i + b \right)
$

Onde:
- **x (entradas):** os dados
- **w (pesos):** importância de cada entrada
- **b (bias/viés):** “ajuste fino” que desloca a ativação
- **f (ativação):** decide como o neurônio responde (degrau, ReLU, sigmoid...)

Problema Real: "Aprova Empréstimo?"

X1 = Tem emprego? (0=Não, 1=Sim)
X2 = Tem renda? (0=Não, 1=Sim)
Y  = Aprova? (0=Nega, 1=Aprova)

Regra do banco: APROVA SÓ se TEMOS AMBOS!

Emprego	Renda	Aprova?

- 0	0	0 ❌
- 0	1	0 ❌
- 1	0	0 ❌
- 1	1	1 ✅


## 2. Prática: Perceptron "Na Unha" (Python Puro)
Vamos criar um neurônio que aprende a lógica **E (AND)**.
*   Entrada 0 e 0 -> Saída 0
*   Entrada 0 e 1 -> Saída 0
*   Entrada 1 e 0 -> Saída 0
*   Entrada 1 e 1 -> Saída 1

In [11]:
class AprovadorEmprestimo:

    def __init__(self):
        """Analista iniciante: não sabe nada ainda"""
        self.peso_emprego = 0.0
        self.peso_renda = 0.0
        self.bias = 0.0

    def calcular_nota(self, emprego, renda):
        """Multiplica cada fator pelo seu peso!"""
        nota_emprego = emprego * self.peso_emprego
        nota_renda = renda * self.peso_renda
        nota_final = nota_emprego + nota_renda + self.bias

        return 1 if nota_final > 0 else 0

    # lr = learning rate (taxa de aprendizado)
    def treinar(self, dados, alvos, epocas=10, lr=0.3):
        print("🏦 === TREINO DO ANALISTA ===\n")

        for epoca in range(epocas):
            erros = 0

            print(f"🔄 ÉPOCA {epoca+1}/{epocas}")

            for i, (emprego, renda) in enumerate(dados):
                alvo = alvos[i]
                predicao = self.calcular_nota(emprego, renda)

                erro = alvo - predicao

                if erro != 0:
                    erros += 1

                # APRENDE com erro!
                self.peso_emprego += lr * erro * emprego
                self.peso_renda += lr * erro * renda
                self.bias += lr * erro

            print(f"📊 Pesos: emprego={self.peso_emprego:.2f}, "
                  f"renda={self.peso_renda:.2f}, bias={self.bias:.2f} | "
                  f"Erros: {erros}")

            if erros == 0:
                print("🎉 ANALISTA VIRA GERENTE! 0 erros!")
                break

            print()


In [12]:
# DADOS DO BANCO - Separar Treino e Teste
X_banco = np.array([[0,0], [0,1], [1,0], [1,1]])
y_banco = np.array([0, 0, 0, 1])

In [13]:
# TREINA
analista = AprovadorEmprestimo()
analista.treinar(X_banco, y_banco)

🏦 === TREINO DO ANALISTA ===

🔄 ÉPOCA 1/10
📊 Pesos: emprego=0.30, renda=0.30, bias=0.30 | Erros: 1

🔄 ÉPOCA 2/10
📊 Pesos: emprego=0.60, renda=0.30, bias=0.00 | Erros: 3

🔄 ÉPOCA 3/10
📊 Pesos: emprego=0.60, renda=0.30, bias=-0.30 | Erros: 3

🔄 ÉPOCA 4/10
📊 Pesos: emprego=0.60, renda=0.60, bias=-0.30 | Erros: 2

🔄 ÉPOCA 5/10
📊 Pesos: emprego=0.60, renda=0.30, bias=-0.60 | Erros: 1

🔄 ÉPOCA 6/10
📊 Pesos: emprego=0.60, renda=0.30, bias=-0.60 | Erros: 0
🎉 ANALISTA VIRA GERENTE! 0 erros!


In [14]:
print(f"\n🏆 Pesos finais:")
print(f"   - Emprego: {analista.peso_emprego:.2f}")
print(f"   - Renda: {analista.peso_renda:.2f}")
print(f"   - Bias (regra banco): {analista.bias:.2f}")


🏆 Pesos finais:
   - Emprego: 0.60
   - Renda: 0.30
   - Bias (regra banco): -0.60


In [15]:
print("=== TESTE FINAL DO BANCO ===\n")
print("Desempregado, sem renda:", analista.calcular_nota(0, 0))
print("Renda OK, mas desempregado:", analista.calcular_nota(0, 1))
print("Emprego OK, mas sem renda:", analista.calcular_nota(1, 0))
print("✅ Empregado + Renda:", analista.calcular_nota(1, 1))

=== TESTE FINAL DO BANCO ===

Desempregado, sem renda: 0
Renda OK, mas desempregado: 0
Emprego OK, mas sem renda: 0
✅ Empregado + Renda: 1


## 3. DEBUB - Perceptron "Na Unha" (Python Puro)


In [16]:
import numpy as np

class AprovadorEmprestimo:
    def __init__(self):
        """Analista iniciante: não sabe nada ainda"""
        self.peso_emprego = 0.0
        self.peso_renda = 0.0
        self.bias = 0.0

    def calcular_nota(self, emprego, renda):
        """Multiplica cada fator pelo seu peso!"""
        nota_emprego = emprego * self.peso_emprego
        nota_renda = renda * self.peso_renda
        nota_final = nota_emprego + nota_renda + self.bias
        return 1 if nota_final > 0 else 0

    def treinar(self, dados, alvos, epocas=10, lr=0.3):
        print("🏦 === TREINO DO ANALISTA (MODO DETALHADO) ===\n")

        for epoca in range(epocas):
            erros = 0
            print(f"=========================================")
            print(f"🔄 INICIANDO ÉPOCA {epoca+1}/{epocas}")
            print(f"=========================================")

            for i, (emprego, renda) in enumerate(dados):
                alvo = alvos[i]

                # Recalculando a nota final apenas para exibir no print
                nota_final_bruta = (emprego * self.peso_emprego) + (renda * self.peso_renda) + self.bias
                predicao = self.calcular_nota(emprego, renda)
                erro = alvo - predicao

                print(f"\n👤 Cliente {i+1}: Emprego={emprego}, Renda={renda} | Alvo Esperado={alvo}")
                print(f"   Cálculo da Nota: ({emprego} * {self.peso_emprego:.2f}) + ({renda} * {self.peso_renda:.2f}) + ({self.bias:.2f}) = {nota_final_bruta:.2f}")
                print(f"   Predição do Neurônio: {predicao} | Erro: {erro} (Alvo {alvo} - Predição {predicao})")

                if erro != 0:
                    erros += 1
                    print("   ⚠️ Erro detectado! Aplicando a regra de aprendizado...")

                    # Guardando os pesos antigos para mostrar na fórmula
                    peso_emp_antigo = self.peso_emprego
                    peso_ren_antigo = self.peso_renda
                    bias_antigo = self.bias

                    # APRENDE com erro!
                    self.peso_emprego += lr * erro * emprego
                    self.peso_renda += lr * erro * renda
                    self.bias += lr * erro

                    print(f"   [Fórmula: Novo = Antigo + (Taxa_Aprendizado * Erro * Entrada)]")
                    print(f"   -> Novo Peso Emprego: {peso_emp_antigo:.2f} + ({lr} * {erro} * {emprego}) = {self.peso_emprego:.2f}")
                    print(f"   -> Novo Peso Renda:   {peso_ren_antigo:.2f} + ({lr} * {erro} * {renda}) = {self.peso_renda:.2f}")
                    print(f"   -> Novo Bias:         {bias_antigo:.2f} + ({lr} * {erro} * 1) = {self.bias:.2f}")
                else:
                    print("   ✅ Acertou! Pesos mantidos sem alteração.")

            print(f"\n📊 RESUMO DA ÉPOCA {epoca+1}: Erros totais = {erros}")
            print(f"Pesos atuais: emprego={self.peso_emprego:.2f}, renda={self.peso_renda:.2f}, bias={self.bias:.2f}\n")

            if erros == 0:
                print("🎉 ANALISTA VIRA GERENTE! 0 erros encontrados na época. Treino concluído!")
                break



In [17]:
# DADOS DO BANCO
X_banco = np.array([[0,0], [0,1], [1,0], [1,1]])
y_banco = np.array([0, 1, 1, 1])

In [18]:
# TREINA
analista = AprovadorEmprestimo()
analista.treinar(X_banco, y_banco)

🏦 === TREINO DO ANALISTA (MODO DETALHADO) ===

🔄 INICIANDO ÉPOCA 1/10

👤 Cliente 1: Emprego=0, Renda=0 | Alvo Esperado=0
   Cálculo da Nota: (0 * 0.00) + (0 * 0.00) + (0.00) = 0.00
   Predição do Neurônio: 0 | Erro: 0 (Alvo 0 - Predição 0)
   ✅ Acertou! Pesos mantidos sem alteração.

👤 Cliente 2: Emprego=0, Renda=1 | Alvo Esperado=1
   Cálculo da Nota: (0 * 0.00) + (1 * 0.00) + (0.00) = 0.00
   Predição do Neurônio: 0 | Erro: 1 (Alvo 1 - Predição 0)
   ⚠️ Erro detectado! Aplicando a regra de aprendizado...
   [Fórmula: Novo = Antigo + (Taxa_Aprendizado * Erro * Entrada)]
   -> Novo Peso Emprego: 0.00 + (0.3 * 1 * 0) = 0.00
   -> Novo Peso Renda:   0.00 + (0.3 * 1 * 1) = 0.30
   -> Novo Bias:         0.00 + (0.3 * 1 * 1) = 0.30

👤 Cliente 3: Emprego=1, Renda=0 | Alvo Esperado=1
   Cálculo da Nota: (1 * 0.00) + (0 * 0.30) + (0.30) = 0.30
   Predição do Neurônio: 1 | Erro: 0 (Alvo 1 - Predição 1)
   ✅ Acertou! Pesos mantidos sem alteração.

👤 Cliente 4: Emprego=1, Renda=1 | Alvo Esperado=